In [ ]:
## Cell 2 — 버전 고정 (원 실행과 동일하게)  → 끝나면 런타임 "세션 다시 시작" 후 Cell 0부터 재실행
!pip -q install "transformers==5.17.0" "accelerate==1.14.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 132.3 MB/s eta 0:00:00


In [ ]:
## Cell 0 — 환경 고정 + 산출물 위치·스키마 확인  (CPU, Drive 해시 포함 L4/A100 2~3분)
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"          # CUDA 초기화 전에 설정
import sys, json, glob, hashlib, platform, subprocess
import numpy as np, pandas as pd, torch, transformers
from google.colab import drive; drive.mount("/content/drive")

torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False

P9  = "/content/drive/MyDrive/Grad_share/data_share/P9/"
OUT = P9 + "steer_v3/"; os.makedirs(OUT + "env", exist_ok=True)

def md5(p, chunk=1 << 24):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

# 1) 환경 기록
ENV = dict(python=platform.python_version(), torch=torch.__version__,
           transformers=transformers.__version__, cuda=torch.version.cuda,
           gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
try:
    import accelerate; ENV["accelerate"] = accelerate.__version__
except ImportError: pass
print("ENV", ENV)
open(OUT + "env/pip_freeze.txt", "w").write(
    subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)

# 2) 데이터 폴더 확인
for d in ["hidden_v2", "p8s_v2"]:
    print(f"{d:10}", os.path.isdir(P9 + d), sorted(os.listdir(P9 + d)) if os.path.isdir(P9 + d) else "")

# 3) 관련 산출물 전수 탐색 (+0.101 런 흔적 포함)
pats = ["**/folds*.json", "**/*steer*", "**/p8s_prompt*.py", "**/k_relabel*",
        "**/p8s_behavior_v2*", "**/*meta.json", "**/*sc*.parquet", "**/*cb_*"]
found = sorted({f for p in pats for f in glob.glob(P9 + p, recursive=True)})
for f in found:
    st = os.stat(f)
    print(f"{st.st_size/1e6:9.2f}MB  {pd.Timestamp(st.st_mtime, unit='s', tz='Asia/Seoul'):%m-%d %H:%M}  {f.replace(P9,'')}")

# 4) 라벨 파일 스키마
beh = [f for f in found if f.endswith("p8s_behavior_v2.csv")]
kr  = [f for f in found if "k_relabel" in f and f.endswith(".parquet")]
if beh:
    m = pd.read_csv(beh[0]); print("\nbehavior", m.shape); print(list(m.columns))
    for c in ["cell_k", "kside_k", "ctx_label", "prop"]:
        if c in m: print(c, m[c].value_counts().to_dict())
if kr:
    q = pd.read_parquet(kr[0]); print("\nk_relabel", q.shape); print(list(q.columns))
    if "cb_samples" in q: print("cb_samples 예시:", q.cb_samples.iloc[0])

# 5) 입력 해시 (hidden은 h_out만)
hid = [f for f in glob.glob(P9 + "**/p8s_v2_h_out.npy", recursive=True)]
MAN = {"env": ENV, "hash": {f.replace(P9, ""): md5(f) for f in beh + kr + hid}}
json.dump(MAN, open(OUT + "env/manifest_cell0.json", "w"), indent=2, ensure_ascii=False)
print("\n", json.dumps(MAN["hash"], indent=1))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ENV {'python': '3.13.15', 'torch': '2.11.0+cu128', 'transformers': '5.17.0', 'cuda': '12.8', 'gpu': 'NVIDIA A100-SXM4-40GB', 'accelerate': '1.14.0'}
hidden_v2  True ['k_flip_audit.csv', 'k_relabel_v2.parquet', 'p8s_behavior_v2.csv', 'p8s_behavior_v2.gsheet', 'p8s_v2_acs.npy', 'p8s_v2_attn_out.npy', 'p8s_v2_h_out.npy', 'p8s_v2_meta.json', 'p8s_v2_mlp_out.npy', 'p8s_v2_mlpnorm.npy', 'p8s_v2_pks.npy', 'p8s_v2_post_attn_ln_weight.npy', 'steer_all.parquet', 'steer_baseline.parquet', 'steer_cplus.parquet', 'steer_v2', 'steer_w.npy']
p8s_v2     False 
     0.62MB  09-16 18:27  E0/e0_scored.parquet
     0.02MB  08-26 14:09  goldfree_v2/steer_ckpt.npz
     0.03MB  08-26 18:54  goldfree_v2/steer_coef_ckpt.npz
     0.08MB  09-16 20:54  hidden_v2/k_relabel_v2.parquet
     1.31MB  09-16 20:54  hidden_v2/p8s_behavior_v2.csv
     0.00MB  09-17 16:43  hidden_v2/p8s_behavior_

In [ ]:
## Cell 1 — 프롬프트 모듈·모델 revision 고정 + folds 확인 + 행별 프롬프트 해시  (CPU, L4/A100 1~2분)
import requests, importlib
from huggingface_hub import HfApi
from transformers import AutoTokenizer

# 1) p8s_prompt.py를 커밋 SHA로 고정
REPO = "Ontology0/Graduation-Project"
SHA  = requests.get(f"https://api.github.com/repos/{REPO}/commits/dev", timeout=20).json()["sha"]
src  = requests.get(f"https://raw.githubusercontent.com/{REPO}/{SHA}/prompt/p8s_prompt.py", timeout=20).text
os.makedirs(OUT + "code", exist_ok=True); open(OUT + "code/p8s_prompt.py", "w").write(src)
sys.path.insert(0, OUT + "code"); import p8s_prompt as PP; importlib.reload(PP)
kr = pd.read_parquet(P9 + "hidden_v2/k_relabel_v2.parquet")
print("prompt:", PP.PROMPT_VERSION, "| commit", SHA[:10])
print("  md5(file)  =", hashlib.md5(src.encode()).hexdigest())
print("  md5(SYSTEM)=", hashlib.md5(PP.SYSTEM_PROMPT.encode()).hexdigest())
print("  k_relabel prompt_md5 값:", kr.prompt_md5.unique()[:3], "| 기록값 20959a4c…")
print("meta.json:", json.load(open(P9 + "hidden_v2/p8s_v2_meta.json")))

# 2) 모델 revision 고정 + chat template 날짜 삽입 여부
MODEL_ID = PP.MODEL_ID
REV = HfApi().model_info(MODEL_ID).sha
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REV)
m = pd.read_csv(P9 + "hidden_v2/p8s_behavior_v2.csv"); m["row_uid"] = np.arange(len(m))
prompts = [tok.apply_chat_template(PP.build_messages(r.question, r.ctx_text),
                                   tokenize=False, add_generation_prompt=True) for r in m.itertuples()]
print("\nmodel revision:", REV)
print(repr(prompts[0][:320]))
today = pd.Timestamp.now(tz="Asia/Seoul")
print("오늘 날짜 삽입:", any(today.strftime(f) in prompts[0] for f in ["%d %b %Y", "%Y-%m-%d", "%B %d, %Y"]))
print("md5(chat_template):", hashlib.md5(tok.chat_template.encode()).hexdigest())

# 3) folds.json 구조 → row 매핑
F = json.load(open(P9 + "hidden_v2/steer_v2/folds.json"))
print("\nfolds.json:", type(F).__name__, list(F)[:5] if isinstance(F, dict) else f"len={len(F)}")
print(json.dumps(F, ensure_ascii=False)[:400])
fold = None
if isinstance(F, dict) and set(m.question) <= set(F):                    # {question: fold}
    fold = m.question.map(F).astype(int).values
elif isinstance(F, dict) and "folds" in F: F = F["folds"]
if fold is None and isinstance(F, list) and all(isinstance(x, list) for x in F):
    qs_ = kr.question.values if max(max(x) for x in F) < len(kr) else None     # 문항 인덱스로 가정
    if qs_ is not None:
        qf = {q: k for k, idx in enumerate(F) for q in qs_[idx]}
        fold = m.question.map(qf).astype(int).values
if fold is not None:
    m["fold"] = fold
    print("\nfold × cell_k\n", pd.crosstab(m.fold, m.cell_k))
    print("같은 질문 C+/C− fold 일치:", (m.groupby("question").fold.nunique() == 1).all())
else:
    print("\n※ 자동 매핑 실패 — 위 구조 출력 보고 맞출게")

# 4) 행 manifest 저장
m["prompt_md5"] = [hashlib.md5(p.encode()).hexdigest() for p in prompts]
m["n_prompt_tok"] = [len(tok(p, add_special_tokens=False)["input_ids"]) for p in prompts]
keep = ["row_uid", "question", "ctx_label", "cell_k", "prop", "prompt_md5", "n_prompt_tok"] + (["fold"] if fold is not None else [])
m[keep].to_parquet(OUT + "env/row_manifest.parquet", index=False)
json.dump({**MAN, "prompt_commit": SHA, "model_revision": REV,
           "chat_template_md5": hashlib.md5(tok.chat_template.encode()).hexdigest()},
          open(OUT + "env/manifest_cell1.json", "w"), indent=2, ensure_ascii=False)
print("n_prompt_tok 중앙값/최대:", int(m.n_prompt_tok.median()), m.n_prompt_tok.max())

prompt: p8s-v2 | commit 209873f9b6
  md5(file)  = 20959a4c34203ea0866d030baf23595d
  md5(SYSTEM)= 67fe2590b5780bad95bdfdcfa0aedf98
  k_relabel prompt_md5 값: ['20959a4c34203ea0866d030baf23595d'] | 기록값 20959a4c…
meta.json: {'prompt_version': 'p8s-v2', 'prompt_md5': '20959a4c34203ea0866d030baf23595d', 'model_id': 'meta-llama/Llama-3.1-8B-Instruct', 'transformers_version': '5.17.0', 'torch_version': '2.11.0+cu128', 'dtype_compute': 'bfloat16', 'dtype_stored': 'float16', 'n_layers': 32, 'hidden_size': 4096, 'batch_size': 16, 'layer_index': '0-based; index i == model.model.layers[i]', 'position': 'last prefill token (left padding, [:, -1, :])', 'row_order': 'p8s_behavior.csv의 행 순서 그대로 (0..1223)', 'streams': ['h_out', 'attn_out', 'mlp_out'], 'reconstruct': {'h_mid': 'h_out[l] - mlp_out[l]', 'h_in': 'h_out[l] - mlp_out[l] - attn_out[l]  (l>=1이면 h_out[l-1]과 일치)', 'mlp_in': 'RMSNorm(h_mid[l]) * post_attn_ln_weight[l], eps=rms_norm_eps'}, 'rms_norm_eps': 1e-05, 'smoke': {'residual_relRMS': 0.0026

In [ ]:
## Cell 3 — 9/19 산출물 확인 + 저장본만으로 결과 재계산 (CPU, 1분)
SV2 = P9 + "hidden_v2/steer_v2/"
fs = sorted(os.listdir(SV2)); print(fs)

m = pd.read_csv(P9 + "hidden_v2/p8s_behavior_v2.csv"); isCm = (m.ctx_label == "C-").values
G = {f[4:-8]: pd.read_parquet(SV2 + f) for f in fs if f.startswith("gen_") and f.endswith(".parquet")}
A = G["base_A"]
print(A[["prompt_version","prompt_md5","tf_version","layers","hook_pos","sigma_def","scorer"]].iloc[0].to_dict())

rows = []
for n, d in G.items():
    assert (d.rid.values == np.arange(1224)).all()
    for c, msk in [("C-", isCm), ("C+", ~isCm)]:
        diff = d.em.values[msk] - A.em.values[msk]
        bs = diff[np.random.RandomState(0).randint(0, len(diff), (5000, len(diff)))].mean(1)
        rows.append(dict(cond=n, C=c, dEM=diff.mean(), lo=np.percentile(bs, 2.5),
                         hi=np.percentile(bs, 97.5), hit=d.hit_limit.values[msk].mean()))
T = pd.DataFrame(rows)
print(T.pivot(index="cond", columns="C", values=["dEM", "lo", "hi"]).round(4).to_string())

MAN["steer_v2_md5"] = {f: md5(SV2 + f) for f in fs}
json.dump(MAN, open(OUT + "env/manifest_cell3.json", "w"), indent=1, ensure_ascii=False)

['audit_w6_changes.csv', 'folds.json', 'gen_base_A.parquet', 'gen_base_B.parquet', 'gen_kdir+6.parquet', 'gen_rand0+6.parquet', 'gen_rand1+6.parquet', 'gen_rand2+6.parquet', 'gen_w+2.parquet', 'gen_w+4.parquet', 'gen_w+6.parquet', 'gen_w+8.parquet', 'gen_w-2.parquet', 'gen_w-4.parquet', 'gen_w-6.parquet', 'gen_wperpK+6.parquet', 'gen_zero_vec.parquet', 'summary.csv', 'w_offline.npz', 'w_offline_info.json']
{'prompt_version': 'p8s-v2', 'prompt_md5': '20959a4c34203ea0866d030baf23595d', 'tf_version': '5.17.0', 'layers': '14-18(0-base)', 'hook_pos': 'last_pos_prefill+decode', 'sigma_def': 'grand_trainfold', 'scorer': 'trunc_abbrfix'}
             dEM              lo              hi        
C             C+      C-      C+      C-      C+      C-
cond                                                    
base_A    0.0000  0.0000  0.0000  0.0000  0.0000  0.0000
base_B    0.0000  0.0000  0.0000  0.0000  0.0000  0.0000
kdir+6    0.0033  0.0997 -0.0114  0.0752  0.0180  0.1242
rand0+6   0.0016  0.

In [ ]:
## Cell 4a — m 복구 (row manifest 기준) + 프롬프트 해시 대조
m   = pd.read_csv(P9 + "hidden_v2/p8s_behavior_v2.csv")
man = pd.read_parquet(OUT + "env/row_manifest.parquet")
assert (man.question.values == m.question.values).all() and (man.ctx_label.values == m.ctx_label.values).all()
m["rid"], m["fold"], m["n_prompt_tok"], m["prompt"] = np.arange(len(m)), man.fold.values, man.n_prompt_tok.values, prompts
assert all(hashlib.md5(p.encode()).hexdigest() == h for p, h in zip(prompts, man.prompt_md5)), "프롬프트 불일치"
print("ok", m.fold.value_counts().sort_index().to_dict())

ok {0: 246, 1: 246, 2: 244, 3: 244, 4: 244}


In [ ]:
## Cell 4 — 재현 테스트: 현재 환경에서 base_A·w+6 재생성 → 9/19 저장본과 문자열 비교
##          (A100 모델 로드 1~2분 + 생성 2조건 약 2분 / L4 약 3분 + 약 4분)
import re, time
from transformers import AutoModelForCausalLM
assert transformers.__version__ == "5.17.0", transformers.__version__
SV2 = P9 + "hidden_v2/steer_v2/"

# --- 9/19 노트북 Cell 0·2와 동일 설정 ---
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
EOS_IDS = sorted({tok.eos_token_id, tok.convert_tokens_to_ids("<|eot_id|>")} - {None})
DECODE_KW = dict(skip_special_tokens=True, clean_up_tokenization_spaces=False)
#model = AutoModelForCausalLM.from_pretrained(PP.MODEL_ID, revision=REV, dtype=torch.bfloat16,
#            device_map="auto", attn_implementation="sdpa").eval()

#m["prompt"], m["rid"] = prompts, m.row_uid
FOLD_ROWS = {k: m[m.fold == k].sort_values(["n_prompt_tok", "rid"], kind="stable").rid.values for k in range(5)}
LAYERS, BS = list(range(14, 19)), 16

class Steer:                                   # 원본과 동일 (마지막 위치, in-place)
    def __init__(self):
        self.vec, self.coef, self.sig, self.calls = None, 0.0, {}, {}
        self.h = [model.model.layers[L].register_forward_hook(self._mk(L)) for L in LAYERS]
    def _mk(self, L):
        def hook(mod, args, out):
            if self.vec is None: return out
            h = out[0] if isinstance(out, (tuple, list)) else out
            self.calls[L] = self.calls.get(L, 0) + 1
            h[:, -1, :] += ((self.coef * self.sig[L]) * self.vec[L]).to(h.dtype)
            return out
        return hook
    def set(self, vec=None, coef=0.0, sig=None):
        self.vec = None if vec is None else {L: torch.as_tensor(vec[L], device=model.device, dtype=model.dtype) for L in LAYERS}
        self.coef, self.sig, self.calls = float(coef), (sig or {}), {}
steer = Steer()

@torch.no_grad()
def gen_fold(k):
    out, rids = {}, FOLD_ROWS[k]
    for s in range(0, len(rids), BS):
        b = rids[s:s+BS]
        enc = tok(m.prompt.values[b].tolist(), return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        g = model.generate(**enc, max_new_tokens=PP.MAX_NEW_TOKENS, do_sample=False,
                           pad_token_id=tok.pad_token_id, eos_token_id=EOS_IDS)[:, enc.input_ids.shape[1]:]
        for r, row in zip(b, g.tolist()):
            stop = next((i for i, t in enumerate(row) if t in EOS_IDS), None)
            n = stop if stop is not None else len(row)
            out[int(r)] = (tok.decode(row[:n], **DECODE_KW), n)
    return out

Z = np.load(SV2 + "w_offline.npz")
W   = {k: {L: Z[f"w_f{k}_L{L}"] for L in LAYERS} for k in range(5)}
SIG = {k: {L: float(Z[f"sig_f{k}_L{L}"]) for L in LAYERS} for k in range(5)}

for name, vec, cf in [("base_A", None, 0.0), ("w+6", W, 6.0)]:
    t0, rows = time.time(), []
    for k in range(5):
        steer.set(None if vec is None else vec[k], cf, None if vec is None else SIG[k])
        o = gen_fold(k)
        if vec is not None: assert all(steer.calls.get(L, 0) > 0 for L in LAYERS)
        rows += [dict(rid=r, raw=x, n_tok=n) for r, (x, n) in o.items()]
    steer.set(None)
    new = pd.DataFrame(rows).sort_values("rid").reset_index(drop=True)
    old = pd.read_parquet(SV2 + f"gen_{name}.parquet")
    new.to_parquet(OUT + f"repro_{name}.parquet", index=False)
    diff = (new.raw.values != old.raw.values)
    print(f"{name:7s} {time.time()-t0:.0f}s  문자열 불일치 {diff.sum()}/1224  (C− {diff[m.ctx_label.values=='C-'].sum()})")
    if diff.any():
        print(pd.DataFrame({"rid": new.rid[diff], "old": old.raw[diff], "new": new.raw[diff]}).head(10).to_string())
print(ENV if "ENV" in dir() else "", torch.cuda.get_device_name(0))

base_A  56s  문자열 불일치 0/1224  (C− 0)
w+6     59s  문자열 불일치 0/1224  (C− 0)
{'python': '3.13.15', 'torch': '2.11.0+cu128', 'transformers': '5.17.0', 'cuda': '12.8', 'gpu': 'NVIDIA A100-SXM4-40GB', 'accelerate': '1.14.0'} NVIDIA A100-SXM4-40GB


In [ ]:
## Cell 5 — 기준 환경 동결 + 골든셋 생성·자기검증 (A100 약 30초 / L4 약 1분)
open(OUT + "env/requirements_lock.txt", "w").write(
    "transformers==5.17.0\naccelerate==1.14.0\n# torch 2.11.0+cu128 (Colab 기본)\n"
    f"# model {PP.MODEL_ID}@{REV}\n# prompt commit {SHA}  md5 20959a4c34203ea0866d030baf23595d\n"
    "# gen: bf16, sdpa, greedy, max_new_tokens=192, bs=16, sort=(n_prompt_tok, rid)\n")

rng = np.random.RandomState(20260921)
GOLD = np.sort(np.concatenate([rng.choice(m.rid[m.cell_k == c].values, 16, replace=False)
                               for c in sorted(m.cell_k.unique())]))          # 셀당 16 = 64행

@torch.no_grad()
def gen_rows(rids):
    rids = m.loc[rids].sort_values(["n_prompt_tok", "rid"], kind="stable").rid.values
    out = {}
    for s in range(0, len(rids), BS):
        b = rids[s:s+BS]
        enc = tok(m.prompt.values[b].tolist(), return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        g = model.generate(**enc, max_new_tokens=PP.MAX_NEW_TOKENS, do_sample=False,
                           pad_token_id=tok.pad_token_id, eos_token_id=EOS_IDS)[:, enc.input_ids.shape[1]:]
        for r, row in zip(b, g.tolist()):
            stop = next((i for i, t in enumerate(row) if t in EOS_IDS), None)
            out[int(r)] = row[: stop if stop is not None else len(row)]
    return out

def golden_run():
    rec = []
    for name, vec, cf in [("base", None, 0.0), ("w+6", W, 6.0)]:
        for k in range(5):
            rk = GOLD[m.fold.values[GOLD] == k]
            steer.set(None if vec is None else vec[k], cf, None if vec is None else SIG[k])
            rec += [dict(cond=name, rid=r, ids=json.dumps(t)) for r, t in gen_rows(rk).items()]
        steer.set(None)
    return pd.DataFrame(rec).sort_values(["cond", "rid"]).reset_index(drop=True)

g1, g2 = golden_run(), golden_run()
print("골든셋 자기 재현 불일치:", (g1.ids != g2.ids).sum(), "/", len(g1))
g1.to_parquet(OUT + "env/golden_ids.parquet", index=False)
print("저장:", md5(OUT + "env/golden_ids.parquet"))

골든셋 자기 재현 불일치: 0 / 128
저장: 095931aa55adadb98dc7aa3e60cedc49


In [ ]:
## Cell 6 — CB 재생성 (기존 K-1 설정 복제: greedy bs32 + 샘플 bs8×5, T=0.7, top_p=0.9, seed 42)
##          raw 저장 + 기존값 재현 검증 + v3 파서 차이  (A100 약 2분 / L4 약 5분)
from collections import Counter
assert steer.vec is None
kr = pd.read_parquet(P9 + "hidden_v2/k_relabel_v2.parquet")
cbp = [tok.apply_chat_template(PP.build_messages_cb(q), tokenize=False, add_generation_prompt=True) for q in kr.question]

@torch.no_grad()
def gen_cb(idx, **kw):
    enc = tok([cbp[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    return model.generate(**enc, max_new_tokens=PP.MAX_NEW_TOKENS, pad_token_id=tok.pad_token_id,
                          eos_token_id=EOS_IDS, use_cache=True, **kw)[:, enc.input_ids.shape[1]:]

dec = lambda x: tok.decode(x, **DECODE_KW)
torch.manual_seed(42); greedy, samp, t0 = [], [], time.time()
for s in range(0, len(cbp), 32):
    idx = list(range(s, min(s + 32, len(cbp))))
    greedy += [dec(x) for x in gen_cb(idx, do_sample=False)]
    for t in range(0, len(idx), 8):
        sub = idx[t:t + 8]
        q = gen_cb(sub, do_sample=True, temperature=0.7, top_p=0.9, num_return_sequences=5)
        samp += [[dec(x) for x in q[j*5:(j+1)*5]] for j in range(len(sub))]
print(f"생성 {time.time()-t0:.0f}s")

# ① 재현 검증: 구 파서로 자른 값이 기존 저장값과 같은가
old_g = kr.cb_greedy.values
old_s = [json.loads(x) if isinstance(x, str) else list(x) for x in kr.cb_samples]
rg = np.array([PP.truncate_answer(x) == o for x, o in zip(greedy, old_g)])
rs = np.array([[PP.truncate_answer(x) for x in f] == o for f, o in zip(samp, old_s)])
print(f"재현  greedy {rg.sum()}/612   samples {rs.sum()}/612")

# ② v3 파서 (= anchor scorer trunc_abbrfix)
ABBR = r"(?<!\bDr)(?<!\bMr)(?<!\bMs)(?<!\bMrs)(?<!\bSt)(?<!\bJr)(?<!\bSr)(?<!\b[A-Z])"
def trunc_v3(raw):
    s = str(raw).strip().split("\n")[0].strip()
    s = re.sub(r"^(answer|the answer is)\s*:?\s*", "", s, flags=re.I)
    if len(s.split()) > 8: s = re.split(ABBR + r"\.\s+(?=[A-Z])", s)[0]
    return re.sub(r"[.,;:\s]+$", "", s.strip().strip('"\'')).strip()

cb = pd.DataFrame({"question": kr.question, "cb_raw": greedy,
                   "cb_samples_raw": [json.dumps(f, ensure_ascii=False) for f in samp]})
cb["cb_greedy_v2"] = [PP.truncate_answer(x) for x in greedy]
cb["cb_greedy_v3"] = [trunc_v3(x) for x in greedy]
cb["cb_greedy_v3_norm"] = cb.cb_greedy_v3.map(PP.norm)
cb["sc_v2"] = [max(Counter(PP.truncate_answer(x) for x in f).values()) / 5 for f in samp]
cb["sc_v3"] = [max(Counter(PP.norm(trunc_v3(x)) for x in f).values()) / 5 for f in samp]
cb["parser"], cb["gen_cfg"] = "trunc_abbrfix", "greedy bs32 | sample T0.7 p0.9 n5 bs8 seed42"
cb.to_parquet(OUT + "cb_v3.parquet", index=False)

ch = cb.cb_greedy_v2 != cb.cb_greedy_v3
print(f"v2→v3 파서로 cb_greedy가 바뀐 문항 {ch.sum()}")
print(cb.loc[ch, ["cb_raw", "cb_greedy_v2", "cb_greedy_v3"]].head(12).to_string())
print(f"자기일치 ≥0.6  v2 {(cb.sc_v2 >= .6).sum()}  v3 {(cb.sc_v3 >= .6).sum()}  (바뀐 문항 {((cb.sc_v2 >= .6) != (cb.sc_v3 >= .6)).sum()})")
print("저장:", md5(OUT + "cb_v3.parquet"))

생성 116s
재현  greedy 579/612   samples 277/612
v2→v3 파서로 cb_greedy가 바뀐 문항 15
                                cb_raw cb_greedy_v2                       cb_greedy_v3
125                            Dr. Dre           Dr                            Dr. Dre
193                            Dr. Dre           Dr                            Dr. Dre
276  Ronald D. Moore and Brannon Braga     Ronald D  Ronald D. Moore and Brannon Braga
304                         T.S. Eliot          T.S                         T.S. Eliot
344                   George H.W. Bush   George H.W                   George H.W. Bush
357                        A.R. Gurney          A.R                        A.R. Gurney
369                            Dr. Dre           Dr                            Dr. Dre
380               William J. Bernstein    William J               William J. Bernstein
382                  Ursula K. Le Guin     Ursula K                  Ursula K. Le Guin
425                     John G. Adolfi       John G    

In [ ]:
## Cell 7 — CB 불일치 진단 (A100 약 30초 / L4 약 1분, greedy만 재실행)
# ① 현재 환경 자기 재현 (greedy만, 같은 배치)
g2 = []
for s in range(0, len(cbp), 32):
    g2 += [dec(x) for x in gen_cb(list(range(s, min(s + 32, len(cbp)))), do_sample=False)]
print("현재 환경 greedy 자기재현:", sum(a == b for a, b in zip(greedy, g2)), "/ 612")

# ② 불일치 33건: 위치·내용·정답 여부
gl = [list(g) if not isinstance(g, str) else json.loads(g) for g in kr.gold_list]
emv = lambda p, g: int(any(PP.norm(p) == PP.norm(x) for x in g))
bad = np.where(~rg)[0]
D = pd.DataFrame({"i": bad, "batch": bad // 32, "old": old_g[bad], "new_v2": cb.cb_greedy_v2.values[bad],
                  "em_old": [emv(old_g[i], gl[i]) for i in bad], "em_new": [emv(cb.cb_greedy_v2[i], gl[i]) for i in bad],
                  "kside_k": kr.kside_k.values[bad], "kside_new": kr.kside_new.values[bad]})
print("배치별 불일치:", D.batch.value_counts().sort_index().to_dict())
print("EM 판정 변화:", int((D.em_old != D.em_new).sum()), "/", len(D))
print(D.to_string())

# ③ 샘플 첫 불일치 위치 (연쇄 여부)
first = int(np.argmin(rs)) if (~rs).any() else None
print("샘플 첫 불일치 index:", first, "| 그 전 greedy 불일치:", int((~rg[:first]).sum()) if first else None)

현재 환경 greedy 자기재현: 612 / 612
배치별 불일치: {0: 1, 3: 1, 4: 3, 5: 1, 6: 1, 7: 1, 10: 1, 11: 1, 12: 5, 13: 2, 14: 1, 15: 3, 16: 3, 17: 7, 18: 2}
EM 판정 변화: 8 / 33
      i  batch                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        old                                                                                                                    

In [ ]:
## Cell 8 — 완화 매칭 재구현 검증 → v3 K± 라벨 (CPU, 즉시)
import unicodedata
fold_ = lambda s: "".join(c for c in unicodedata.normalize("NFKD", PP.norm(s)) if not unicodedata.combining(c))
def relaxed(p, golds):
    p = fold_(p)
    if not p: return 0
    return int(any((g := fold_(x)) and (p == g or g in p or p in g) for x in golds))

# ① 재구현 검증: 구 cb_greedy → kside_k 재현되나
k_old = np.where([relaxed(p, g) for p, g in zip(old_g, gl)], "K+", "K−")
mm = k_old != kr.kside_k.values
print(f"kside_k 재현 {612 - mm.sum()}/612")
if mm.any(): print(pd.DataFrame({"pred": old_g[mm], "gold": [gl[i][:3] for i in np.where(mm)[0]], "kside_k": kr.kside_k.values[mm]}).to_string())

# ② v3 라벨 (고정 환경 greedy + v3 파서)
cb["kside_v3"]   = np.where([relaxed(p, g) for p, g in zip(cb.cb_greedy_v3, gl)], "K+", "K−")
cb["kside_v3em"] = np.where([emv(p, g) for p, g in zip(cb.cb_greedy_v3, gl)], "K+", "K−")
cb["k_agree_v3"] = [np.mean([emv(trunc_v3(x), g) for x in json.loads(f)]) for f, g in zip(cb.cb_samples_raw, gl)]
print("\nkside_k → kside_v3\n", pd.crosstab(kr.kside_k, cb.kside_v3))
print("K+ 수  kside_k", (kr.kside_k == "K+").sum(), "| v3", (cb.kside_v3 == "K+").sum(), "| v3(EM)", (cb.kside_v3em == "K+").sum())
cb.to_parquet(OUT + "cb_v3.parquet", index=False); print("저장:", md5(OUT + "cb_v3.parquet"))

kside_k 재현 612/612

kside_k → kside_v3
 kside_v3   K+   K−
kside_k           
K+        342    4
K−          4  262
K+ 수  kside_k 346 | v3 346 | v3(EM) 319
저장: 159c216ac1ae7983e3d3bce49dcf5d64


In [ ]:
## Cell 9 — v3 w 라벨 집계 + w_v3 추정 + legacy w 대비 (CPU, 1~2분)
H  = np.load(P9 + "hidden_v2/p8s_v2_h_out.npy", mmap_mode="r")
A  = pd.read_parquet(SV2 + "gen_base_A.parquet")
Lb = m[["rid", "question", "ctx_label", "prop", "fold", "answer_surface"]].copy()
Lb["kside_v3"] = Lb.question.map(dict(zip(cb.question, cb.kside_v3)))
p = np.array([PP.norm(trunc_v3(x)) for x in A.raw])
s = Lb.answer_surface.map(PP.norm).values
c = Lb.question.map(dict(zip(cb.question, cb.cb_greedy_v3))).map(PP.norm).values
Lb["acc_x"] = ((s != "") & (p == s)).astype(int)
Lb["ret_x"] = ((c != "") & (c != s) & (p == c)).astype(int)
Lb["cb_eq_surf"] = (c == s).astype(int)
Lb["grp"] = np.select([Lb.ret_x == 1, Lb.acc_x == 1], ["retain", "accept"], "neither")
isC = (Lb.ctx_label == "C-").values
C_ = Lb[isC]

# ① 표본 수 공개 (피드백 요구)
print("legacy→v3 라벨 변경:", int(((A.ret_x.values != Lb.ret_x.values) | (A.acc_x.values != Lb.acc_x.values))[isC].sum()), "행",
      "| cb==surface(제외) C−:", int(C_.cb_eq_surf.sum()))
print(pd.crosstab([C_.fold, C_.kside_v3], C_.grp, margins=True))
PT = pd.crosstab(C_.prop, C_.grp)
PT["used(≥3/3)"] = (PT.get("retain", 0) >= 3) & (PT.get("accept", 0) >= 3)
print(PT.sort_values("retain"))

# ② w_v3 (fold별, train fold C−만, prop 균형 min_n=3) + σ 재계산 검증
LAYERS = list(range(14, 19))
HL = {L: np.asarray(H[:, L]).astype(np.float32) for L in LAYERS}
PR = Lb.prop.values
def prop_bal_diff(X, pos, neg, min_n=3):
    ds, used = [], []
    for q in np.unique(PR[pos | neg]):
        i = PR == q
        if (pos & i).sum() >= min_n and (neg & i).sum() >= min_n:
            ds.append(X[pos & i].mean(0) - X[neg & i].mean(0)); used.append(q)
    d = np.mean(ds, 0); return d / np.linalg.norm(d), used

W3, INFO3 = {}, []
for k in range(5):
    tr = (Lb.fold != k).values & isC
    pos, neg = tr & (Lb.ret_x == 1).values, tr & (Lb.acc_x == 1).values
    W3[k] = {L: prop_bal_diff(HL[L], pos, neg)[0] for L in LAYERS}
    sig = {L: float(HL[L][(Lb.fold != k).values].std()) for L in LAYERS}
    assert all(abs(sig[L] - SIG[k][L]) < 1e-6 for L in LAYERS), "σ 불일치"
    INFO3.append(dict(fold=k, n_retain=int(pos.sum()), n_accept=int(neg.sum()),
                      props=",".join(prop_bal_diff(HL[16], pos, neg)[1]),
                      **{f"cos_legacy_L{L}": round(float(W3[k][L] @ W[k][L]), 4) for L in LAYERS}))
print(pd.DataFrame(INFO3).drop(columns="props").to_string(index=False))

np.savez(OUT + "w_v3.npz", **{f"w_f{k}_L{L}": W3[k][L] for k in range(5) for L in LAYERS},
         **{f"sig_f{k}_L{L}": SIG[k][L] for k in range(5) for L in LAYERS})
Lb.to_parquet(OUT + "labels_v3.parquet", index=False)
json.dump(INFO3, open(OUT + "w_v3_info.json", "w"), ensure_ascii=False, indent=1)
print("저장:", md5(OUT + "w_v3.npz"), md5(OUT + "labels_v3.parquet"))

legacy→v3 라벨 변경: 4 행 | cb==surface(제외) C−: 0
grp            accept  neither  retain  All
fold kside_v3                              
0    K+            22       17      30   69
     K−            35       18       1   54
1    K+            18       24      26   68
     K−            32       20       3   55
2    K+            38        6      24   68
     K−            30       22       2   54
3    K+            31       10      28   69
     K−            32       21       0   53
4    K+            24       16      32   72
     K−            29       18       3   50
All               291      172     149  612
grp             accept  neither  retain  used(≥3/3)
prop                                               
color                2        3       0       False
mother               6        2       1       False
place of birth      30        6       3        True
religion             2       13       3       False
occupation          12       11       4        True
screenwriter       

In [ ]:
## Cell 10 — 사전등록 저장 → v3 anchor (w_v3 +6) 생성·채점  (A100 약 1분 / L4 약 2.5분)
import datetime as dt
PREREG = {
 "date": dt.datetime.now(dt.timezone(dt.timedelta(hours=9))).isoformat(),
 "hashes": {"folds": md5(SV2 + "folds.json"), "cb_v3": md5(OUT + "cb_v3.parquet"),
            "labels_v3": md5(OUT + "labels_v3.parquet"), "w_v3": md5(OUT + "w_v3.npz"),
            "golden": md5(OUT + "env/golden_ids.parquet")},
 "env": "transformers 5.17.0 / accelerate 1.14.0 / model@" + REV + " / bf16 sdpa greedy bs16 sort(n_prompt_tok,rid)",
 "parser_scorer": "trunc_abbrfix / EM(norm exact)",  "K_label": "kside_v3 (relaxed, cb_v3 greedy); kside_k = sensitivity",
 "intervention_primary": "L14-18 0-base, last position (prefill last + every decode step), coef +6 x grand sigma(train fold), w_v3 per fold",
 "primary_endpoint": {"efficacy": "C- dEM, 95% question bootstrap CI lower > 0",
                      "safety": "C+ dEM, 95% CI lower > -0.02 (one-sided alpha .025); -0.01 sensitivity",
                      "legacy_primary_reported": "1224-row EM (original prereg 9/19)",
                      "change_reason": "1:1 C+/C- weighting can offset C- gain with C+ loss (external review 9/21), set before any v3 result"},
 "gates": {"gate_GF": "L13 h_out, train-fold scaler, dir = mean(SC>=0.6) - mean(SC<0.6), tau = midpoint of the two train centroids' projections",
           "gate_EM": "tau by nested inner CV maximizing EM (gold-calibrated upper reference)",
           "gate_Kref": "kside_v3 (noisy behavioral K label reference)", "oracle_outcome": "per-row best of (a)/(b), unattainable ceiling",
           "random_gate": "same open rate, 2000 permutations", "SC_def": "sc_v3 = majority share of 5 samples (T0.7 p0.9 seed42), threshold 0.6 = SHIFT 3/5"},
 "direction_specificity": "-w; permutation-w x10 (labels shuffled within fold x prop); isotropic random x3; claim only if w effect > 95th pct of null",
 "stability": "fail if hit_limit > 0.05 or rep_rate > 0.05 (rep = any word 3-gram occurring >=3 times); failures reported, not removed",
 "sensitivity_preregistered": ["w n-weighted prop average", "kside_k labels", "exclude 0.2<k_agree_v3<0.8"],
 "known_limits": ["retain group 94% K+ -> w confounded with K direction; Exp3 w_K+ tests behavior-only direction",
                  "L14-18 / coef / prompt were developed on p8s -> p8s = development set; external replication required"],
}
json.dump(PREREG, open(OUT + "prereg_v3.json", "w"), ensure_ascii=False, indent=1)
print("사전등록 저장:", md5(OUT + "prereg_v3.json"))

# --- v3 anchor 생성 ---
def rep3(t):
    w = str(t).split(); g = Counter(tuple(w[i:i+3]) for i in range(len(w) - 2))
    return int(bool(g) and max(g.values()) >= 3)
cbq = dict(zip(cb.question, cb.cb_greedy_v3))
def score_v3(d):
    p = [PP.norm(trunc_v3(x)) for x in d.raw]
    s, c = m.answer_surface.map(PP.norm).values, m.question.map(cbq).map(PP.norm).values
    d["pred"] = [trunc_v3(x) for x in d.raw]
    d["em"]    = [int(any(pi == PP.norm(g) for g in gg)) for pi, gg in zip(p, gl_row)]
    d["acc_x"] = [int(si != "" and pi == si) for pi, si in zip(p, s)]
    d["ret_x"] = [int(ci != "" and ci != si and pi == ci) for pi, si, ci in zip(p, s, c)]
    d["hit_limit"] = (d.n_tok >= PP.MAX_NEW_TOKENS).astype(int); d["rep3"] = d.raw.map(rep3)
    return d
gl_row = [json.loads(g) if isinstance(g, str) else list(g) for g in m.gold]

t0, rows = time.time(), []
for k in range(5):
    steer.set(W3[k], 6.0, SIG[k]); o = gen_fold(k)
    assert all(steer.calls.get(L, 0) > 0 for L in LAYERS)
    rows += [dict(rid=r, raw=x, n_tok=n) for r, (x, n) in o.items()]
steer.set(None)
V = score_v3(pd.DataFrame(rows).sort_values("rid").reset_index(drop=True)); V["cond"] = "v3_w+6"
B = score_v3(A[["rid", "raw", "n_tok"]].copy()); B["cond"] = "v3_base"
V.to_parquet(OUT + "gen_v3_w+6.parquet", index=False); B.to_parquet(OUT + "gen_v3_base.parquet", index=False)
print(f"생성 {time.time()-t0:.0f}s | legacy w+6와 문자열 불일치 {(V.raw.values != pd.read_parquet(SV2 + 'gen_w+6.parquet').raw.values).sum()}/1224")

isCm = (m.ctx_label == "C-").values
for c_, msk in [("C-", isCm), ("C+", ~isCm)]:
    d = V.em.values[msk] - B.em.values[msk]
    bs = d[np.random.RandomState(0).randint(0, len(d), (5000, len(d)))].mean(1)
    up, dn = int(((B.em.values[msk] == 0) & (V.em.values[msk] == 1)).sum()), int(((B.em.values[msk] == 1) & (V.em.values[msk] == 0)).sum())
    print(f"{c_}  base {B.em.values[msk].mean():.4f} → {V.em.values[msk].mean():.4f}  ΔEM {d.mean():+.4f} "
          f"[{np.percentile(bs, 2.5):+.4f}, {np.percentile(bs, 97.5):+.4f}]  +{up}/-{dn}  "
          f"hit {V.hit_limit.values[msk].mean():.3f}  rep3 {V.rep3.values[msk].mean():.3f}")

사전등록 저장: 06a8fbcb1eb887c77cbab66648f4860b
생성 60s | legacy w+6와 문자열 불일치 26/1224
C-  base 0.2598 → 0.3595  ΔEM +0.0997 [+0.0752, +0.1242]  +63/-2  hit 0.000  rep3 0.000
C+  base 0.8023 → 0.8072  ΔEM +0.0049 [-0.0114, +0.0212]  +16/-13  hit 0.000  rep3 0.000


In [ ]:
## Cell 11 — Exp1: gate_GF / K-label ref / outcome ceiling / random gate + TE 표 (CPU, 1~2분)
PREREG["gate_training_rows"] = "outer-train fold, both C+ and C- rows (C unknown at deployment); fixed before gate results"
json.dump(PREREG, open(OUT + "prereg_v3.json", "w"), ensure_ascii=False, indent=1)

H13 = np.asarray(H[:, 13]).astype(np.float32)
sc_row = m.question.map(dict(zip(cb.question, cb.sc_v3))).values
ysc = (sc_row >= 0.6).astype(int)
fold = m.fold.values; q = m.question.values
a_em, b_em = B.em.values, V.em.values

# gate_GF: fold별 scaler·방향·τ 전부 outer-train에서
gscore, gopen = np.zeros(1224), np.zeros(1224, bool)
for k in range(5):
    tr, te = fold != k, fold == k
    mu, sd = H13[tr].mean(0), H13[tr].std(0) + 1e-6
    Z = (H13 - mu) / sd
    cdir = Z[tr & (ysc == 1)].mean(0) - Z[tr & (ysc == 0)].mean(0); cdir /= np.linalg.norm(cdir)
    tau = 0.5 * (Z[tr & (ysc == 1)].mean(0) @ cdir + Z[tr & (ysc == 0)].mean(0) @ cdir)
    gscore[te] = Z[te] @ cdir; gopen[te] = gscore[te] > tau

kref = (m.question.map(dict(zip(cb.question, cb.kside_v3))).values == "K+")
POL = {"(a) none": np.zeros(1224, bool), "(b) always": np.ones(1224, bool),
       "gate_GF": gopen, "gate_Kref": kref}
em_of = lambda op: np.where(op, b_em, a_em)

uq = np.unique(q); qidx = {u: np.where(q == u)[0] for u in uq}
rs_ = np.random.RandomState(0); BOOT = [np.concatenate([qidx[u] for u in rs_.choice(uq, len(uq))]) for _ in range(2000)]
def ci_of(v, msk):
    return np.percentile([v[i][msk[i]].mean() for i in BOOT], [2.5, 97.5])

isCm = (m.ctx_label == "C-").values
rows = []
for nm, op in POL.items():
    e = em_of(op)
    r = dict(policy=nm, open=op.mean(), open_Cm=op[isCm].mean(), open_Cp=op[~isCm].mean(), EM_all=e.mean())
    for c_, msk in [("C-", isCm), ("C+", ~isCm), ("all", np.ones(1224, bool))]:
        d = e - a_em; lo, hi = ci_of(d, msk)
        r[f"d_{c_}"], r[f"lo_{c_}"] = d[msk].mean(), lo
    db = e - b_em; r["vs_b_all"], (r["vs_b_lo"], r["vs_b_hi"]) = db.mean(), ci_of(db, np.ones(1224, bool))
    rows.append(r)
ceil = np.maximum(a_em, b_em)
rows.append(dict(policy="oracle_outcome (ceiling)", EM_all=ceil.mean(), d_all=(ceil - a_em).mean(),
                 **{f"d_{c_}": (ceil - a_em)[msk].mean() for c_, msk in [("C-", isCm), ("C+", ~isCm)]}))
T1 = pd.DataFrame(rows); print(T1.round(4).to_string(index=False))

# random gate: gate_GF와 같은 개방 수
rng = np.random.RandomState(1); n_open = gopen.sum()
null = np.array([em_of(np.isin(np.arange(1224), rng.choice(1224, n_open, replace=False))).mean() for _ in range(2000)])
print(f"\ngate_GF EM {em_of(gopen).mean():.4f} | random gate 동일개방 {null.mean():.4f} "
      f"[95th {np.percentile(null, 95):.4f}]  percentile {(null < em_of(gopen).mean()).mean():.3f}")

# treatment effect by gate score (C별 5분위)
TE = b_em - a_em
print("\nTE(b−a) by gate_GF 점수 5분위")
for c_, msk in [("C-", isCm), ("C+", ~isCm)]:
    qq = pd.qcut(gscore[msk], 5, labels=False)
    t = pd.DataFrame({"q": qq, "te": TE[msk]}).groupby("q").te.agg(
        n="size", improved=lambda x: (x == 1).sum(), worsened=lambda x: (x == -1).sum(), net="sum")
    print(c_, t.T.to_string())
print(f"\ngate_GF 진단 AUROC vs kside_v3 = {roc_auc_score(kref, gscore):.3f}")

pd.DataFrame({"rid": m.rid, "gscore": gscore, "gate_GF": gopen, "gate_Kref": kref}).to_parquet(OUT + "exp1_gates.parquet", index=False)
T1.to_csv(OUT + "exp1_summary.csv", index=False)

                  policy   open  open_Cm  open_Cp  EM_all   d_C-  lo_C-   d_C+   lo_C+  d_all  lo_all  vs_b_all  vs_b_lo  vs_b_hi
                (a) none 0.0000   0.0000   0.0000  0.5310 0.0000 0.0000 0.0000  0.0000 0.0000  0.0000   -0.0523  -0.0678  -0.0376
              (b) always 1.0000   1.0000   1.0000  0.5833 0.0997 0.0768 0.0049 -0.0131 0.0523  0.0376    0.0000   0.0000   0.0000
                 gate_GF 0.5319   0.5278   0.5359  0.5621 0.0539 0.0359 0.0082 -0.0033 0.0310  0.0204   -0.0212  -0.0319  -0.0106
               gate_Kref 0.5654   0.5654   0.5654  0.5768 0.0833 0.0621 0.0082 -0.0033 0.0458  0.0335   -0.0065  -0.0147   0.0016
oracle_outcome (ceiling)    NaN      NaN      NaN  0.5956 0.1029    NaN 0.0261     NaN 0.0645     NaN       NaN      NaN      NaN

gate_GF EM 0.5621 | random gate 동일개방 0.5589 [95th 0.5654]  percentile 0.755

TE(b−a) by gate_GF 점수 5분위
C- q           0    1    2    3    4
n         123  122  122  122  123
improved    8   14   16   10   15
worsened   

NameError: name 'roc_auc_score' is not defined

In [ ]:
## Cell 12 — Exp1 저장 + 방향 특이성 (C− 612행, +6): base·w_v3 / −w / perm-w×10 / random×3
##          16조건 × 612행: A100 약 8분 / L4 약 18분 (조건별 체크포인트)
from sklearn.metrics import roc_auc_score
print(f"gate_GF 진단 AUROC vs kside_v3 = {roc_auc_score(kref, gscore):.3f}")
pd.DataFrame({"rid": m.rid, "gscore": gscore, "gate_GF": gopen, "gate_Kref": kref}).to_parquet(OUT + "exp1_gates.parquet", index=False)
T1.to_csv(OUT + "exp1_summary.csv", index=False)

CM_ROWS = {k: m[(m.fold == k) & isCm].sort_values(["n_prompt_tok", "rid"], kind="stable").rid.values for k in range(5)}

@torch.no_grad()
def gen_list(rids):
    out = {}
    for s in range(0, len(rids), BS):
        b = rids[s:s+BS]
        enc = tok(m.prompt.values[b].tolist(), return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        g = model.generate(**enc, max_new_tokens=PP.MAX_NEW_TOKENS, do_sample=False,
                           pad_token_id=tok.pad_token_id, eos_token_id=EOS_IDS)[:, enc.input_ids.shape[1]:]
        for r, row in zip(b, g.tolist()):
            stop = next((i for i, t in enumerate(row) if t in EOS_IDS), None)
            n = stop if stop is not None else len(row)
            out[int(r)] = (tok.decode(row[:n], **DECODE_KW), n)
    return out

def run_cm(name, vec, coef=6.0):
    f = OUT + f"spec_{name}.parquet"
    if os.path.exists(f): return pd.read_parquet(f)
    rows, t0 = [], time.time()
    for k in range(5):
        steer.set(None if vec is None else vec[k], coef, None if vec is None else SIG[k])
        o = gen_list(CM_ROWS[k])
        if vec is not None: assert all(steer.calls.get(L, 0) > 0 for L in LAYERS)
        rows += [dict(rid=r, raw=x, n_tok=n) for r, (x, n) in o.items()]
    steer.set(None)
    d = pd.DataFrame(rows).sort_values("rid").reset_index(drop=True)
    d["em"] = [int(any(PP.norm(trunc_v3(x)) == PP.norm(g) for g in gl_row[r])) for r, x in zip(d.rid, d.raw)]
    d["hit_limit"] = (d.n_tok >= PP.MAX_NEW_TOKENS).astype(int); d["cond"] = name
    d.to_parquet(f, index=False); print(f"{name} {time.time()-t0:.0f}s"); return d

def perm_w(seed):                      # fold×prop 안에서 retain/accept 라벨 셔플 → 같은 절차로 w
    rng, out = np.random.RandomState(seed), {}
    ret, acc = (Lb.ret_x == 1).values, (Lb.acc_x == 1).values
    for k in range(5):
        lab = (Lb.fold != k).values & isC & (ret | acc); pos = np.zeros(1224, bool)
        for j in set(range(5)) - {k}:
            for p_ in np.unique(PR[lab]):
                i = np.where(lab & (PR == p_) & (Lb.fold == j).values)[0]
                if len(i): y = ret[i].copy(); rng.shuffle(y); pos[i] = y
        out[k] = {L: prop_bal_diff(HL[L], pos, lab & ~pos)[0] for L in LAYERS}
    return out

rr = np.random.RandomState(12345)
RAND = [{L: (v := rr.randn(4096).astype(np.float32)) / np.linalg.norm(v) for L in LAYERS} for _ in range(3)]
CONDS = {"base": None, "w_v3+6": W3, "neg_w+6": {k: {L: -W3[k][L] for L in LAYERS} for k in range(5)}}
CONDS.update({f"perm{s}+6": perm_w(s) for s in range(10)})
CONDS.update({f"rand{s}+6": {k: RAND[s] for k in range(5)} for s in range(3)})
D = {n: run_cm(n, v) for n, v in CONDS.items()}

b0 = D["base"].em.values
print("C−전용 배치 base vs 전체배치 base 문자열 불일치:", (D["base"].raw.values != B.raw.values[D["base"].rid.values]).sum())
dl = {n: (d.em.values - b0).mean() for n, d in D.items() if n != "base"}
null = np.array([v for n, v in dl.items() if n.startswith(("perm", "rand"))])
print(pd.Series(dl).round(4).to_string())
print(f"\nw_v3 {dl['w_v3+6']:+.4f} | null(perm10+rand3) mean {null.mean():+.4f}  max {null.max():+.4f}  "
      f"95th {np.percentile(null, 95):+.4f} | w > null 전부: {(dl['w_v3+6'] > null).all()}")
print(f"cos(perm-w, w_v3) L16 fold0:", [round(float(CONDS[f'perm{s}+6'][0][16] @ W3[0][16]), 3) for s in range(10)])

gate_GF 진단 AUROC vs kside_v3 = 0.759
base 29s
w_v3+6 29s
neg_w+6 30s
perm0+6 29s
perm1+6 29s
perm2+6 62s
perm3+6 29s
perm4+6 29s
perm5+6 29s
perm6+6 28s
perm7+6 40s
perm8+6 28s
perm9+6 29s
rand0+6 28s
rand1+6 29s
rand2+6 28s
C−전용 배치 base vs 전체배치 base 문자열 불일치: 5
w_v3+6     0.1013
neg_w+6   -0.1373
perm0+6    0.0556
perm1+6    0.0621
perm2+6    0.0327
perm3+6    0.0474
perm4+6    0.0523
perm5+6    0.0180
perm6+6    0.0458
perm7+6    0.0735
perm8+6    0.0245
perm9+6    0.0180
rand0+6    0.0049
rand1+6    0.0082
rand2+6    0.0082

w_v3 +0.1013 | null(perm10+rand3) mean +0.0347  max +0.0735  95th +0.0667 | w > null 전부: True
cos(perm-w, w_v3) L16 fold0: [0.403, 0.633, 0.431, 0.272, 0.053, 0.185, 0.502, 0.545, 0.698, -0.093]


In [ ]:
## Cell 13 — Exp2: 위치(P9 all / P10 last / CF decode) × ρ 스윕, 1224행
##          19조건: A100 약 20~30분 / L4 약 50~70분 (고강도 붕괴 시 늘어남, 조건별 체크포인트)
PREREG["exp2"] = {
 "dose": "Delta_L = rho * median_train||h_L|| (last prefill token) * w_v3; legacy coef runs kept separate",
 "rho_grid": [0.03, 0.06, 0.09, 0.12, 0.15, 0.20, 0.25], "cf_grid": [0.09, 0.15, 0.25], "p9_noBOS": [0.09],
 "positions": {"P10_last": "prefill last + decode", "P9_all": "all non-pad prefill + decode", "CF_decode": "decode only (first answer token unsteered)"},
 "dose_accounting": "N_inj, D_sum, D_energy reported as bookkeeping proxies, not causal totals",
 "nested_selection": "for held-out fold k: on other 4 folds pick rho maximizing C- EM s.t. C+ dEM >= -0.02; evaluate on k",
 "interpretation": "if P10 reaches P9's peak at a higher safe rho, P9's advantage is not unique to all-token position; total dose not established as cause"}
json.dump(PREREG, open(OUT + "prereg_v3.json", "w"), ensure_ascii=False, indent=1)
print("사전등록 갱신:", md5(OUT + "prereg_v3.json"))

for h_ in steer.h: h_.remove()                     # 기존 훅 제거
BOS = tok.bos_token_id
class Steer2:
    def __init__(self):
        self.vec, self.mode, self.scale, self.mask, self.calls = None, "last", {}, None, {}
        self.h = [model.model.layers[L].register_forward_hook(self._mk(L)) for L in LAYERS]
    def _mk(self, L):
        def hook(mod, args, out):
            if self.vec is None: return out
            h = out[0] if isinstance(out, (tuple, list)) else out
            d = (self.scale[L] * self.vec[L]).to(h.dtype)
            self.calls[L] = self.calls.get(L, 0) + 1
            if h.shape[1] > 1:                                  # prefill
                if self.mode == "last": h[:, -1, :] += d
                elif self.mode in ("all", "all_noBOS"): h += self.mask[:, :, None].to(h.dtype) * d
            else: h[:, -1, :] += d                              # decode (모든 모드)
            return out
        return hook
    def set(self, vec=None, scale=None, mode="last"):
        self.vec = None if vec is None else {L: torch.as_tensor(vec[L], device=model.device, dtype=model.dtype) for L in LAYERS}
        self.scale, self.mode, self.calls = (scale or {}), mode, {}
st2 = Steer2()

@torch.no_grad()
def gen_fold2(k):
    out, rids = {}, FOLD_ROWS[k]
    for s in range(0, len(rids), BS):
        b = rids[s:s+BS]
        enc = tok(m.prompt.values[b].tolist(), return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        mk = enc.attention_mask.clone()
        if st2.mode == "all_noBOS": mk[enc.input_ids == BOS] = 0
        st2.mask = mk
        g = model.generate(**enc, max_new_tokens=PP.MAX_NEW_TOKENS, do_sample=False,
                           pad_token_id=tok.pad_token_id, eos_token_id=EOS_IDS)[:, enc.input_ids.shape[1]:]
        for r, row, npv in zip(b, g.tolist(), enc.attention_mask.sum(1).tolist()):
            stop = next((i for i, t in enumerate(row) if t in EOS_IDS), None)
            n = stop if stop is not None else len(row)
            out[int(r)] = (tok.decode(row[:n], **DECODE_KW), n, npv)
    return out

MED = {k: {L: float(np.median(np.linalg.norm(HL[L][(m.fold != k).values], axis=1))) for L in LAYERS} for k in range(5)}
def run2(name, mode, scale_fn):
    f = OUT + f"exp2_{name}.parquet"
    if os.path.exists(f): return pd.read_parquet(f)
    rows, t0 = [], time.time()
    for k in range(5):
        st2.set(W3[k], {L: scale_fn(k, L) for L in LAYERS}, mode); o = gen_fold2(k)
        assert all(st2.calls.get(L, 0) > 0 for L in LAYERS)
        rows += [dict(rid=r, raw=x, n_tok=n, n_prompt=npv, fold=k, scale_L16=scale_fn(k, 16)) for r, (x, n, npv) in o.items()]
    st2.set(None)
    d = score_v3(pd.DataFrame(rows).sort_values("rid").reset_index(drop=True)); d["cond"], d["mode"] = name, mode
    d.to_parquet(f, index=False)
    print(f"{name:14s} {time.time()-t0:4.0f}s  C− {d.em.values[isCm].mean():.4f}  C+ {d.em.values[~isCm].mean():.4f}  "
          f"hit {d.hit_limit.mean():.3f}  rep3 {d.rep3.mean():.3f}")
    return d

# ① 훅 재검증: legacy +6 (P10 last, 6σ) → gen_v3_w+6과 완전 일치해야 함
chk = run2("chk_legacy6", "last", lambda k, L: 6.0 * SIG[k][L])
print("Steer2 재현 불일치:", (chk.raw.values != V.raw.values).sum(), "/ 1224"); assert (chk.raw.values == V.raw.values).all()

# ② 스윕
E2 = {}
for rho in PREREG["exp2"]["rho_grid"]:
    for mode, tag in [("last", "P10"), ("all", "P9")]:
        E2[f"{tag}_r{rho:.2f}"] = run2(f"{tag}_r{rho:.2f}", mode, lambda k, L, r=rho: r * MED[k][L])
for rho in PREREG["exp2"]["cf_grid"]:
    E2[f"CF_r{rho:.2f}"] = run2(f"CF_r{rho:.2f}", "decode", lambda k, L, r=rho: r * MED[k][L])
E2["P9noBOS_r0.09"] = run2("P9noBOS_r0.09", "all_noBOS", lambda k, L: 0.09 * MED[k][L])
print("MED fold0:", {L: round(MED[0][L], 2) for L in LAYERS}, "| legacy +6 ≈ ρ", round(6 * SIG[0][16] / MED[0][16], 4))

사전등록 갱신: 1c984cecb93d5f6fedd2b576ed5c541d
chk_legacy6      60s  C− 0.3595  C+ 0.8072  hit 0.000  rep3 0.000
Steer2 재현 불일치: 0 / 1224
P10_r0.03        55s  C− 0.3121  C+ 0.8039  hit 0.000  rep3 0.000
P9_r0.03         55s  C− 0.3431  C+ 0.8023  hit 0.000  rep3 0.000
P10_r0.06        54s  C− 0.3431  C+ 0.7990  hit 0.000  rep3 0.000
P9_r0.06         59s  C− 0.3954  C+ 0.8039  hit 0.000  rep3 0.000
P10_r0.09        61s  C− 0.3627  C+ 0.8056  hit 0.000  rep3 0.000
P9_r0.09         54s  C− 0.4265  C+ 0.7745  hit 0.000  rep3 0.000
P10_r0.12        70s  C− 0.3840  C+ 0.7958  hit 0.001  rep3 0.001
P9_r0.12         82s  C− 0.4216  C+ 0.7320  hit 0.002  rep3 0.002
P10_r0.15        89s  C− 0.3856  C+ 0.7810  hit 0.002  rep3 0.002
P9_r0.15        133s  C− 0.4020  C+ 0.6536  hit 0.007  rep3 0.007
P10_r0.20       125s  C− 0.3824  C+ 0.7173  hit 0.005  rep3 0.005
P9_r0.20        342s  C− 0.3448  C+ 0.4461  hit 0.023  rep3 0.023
P10_r0.25       280s  C− 0.3676  C+ 0.5784  hit 0.020  rep3 0.020
P9_r0.25  

In [ ]:
## Cell 14 — Exp2 분석: 조건표 · nested 선택 · 게이트 재평가 (CPU, 1~2분)
E2 = {n: d for n, d in E2.items()}
kv = m.question.map(dict(zip(cb.question, cb.kside_v3))).values
cellv = pd.Series(kv).str.cat(m.ctx_label.values).values
a_em, fo = B.em.values, m.fold.values
ci_d = lambda d, msk: np.percentile([d[i][msk[i]].mean() for i in BOOT], [2.5, 97.5])

# ① 조건표
rows = []
for n, d in E2.items():
    mode, rho = d["mode"].iloc[0], float(n.split("_r")[1]); diff = d.em.values - a_em
    r = dict(cond=n, rho=rho)
    for c_, msk in [("C-", isCm), ("C+", ~isCm)]:
        lo, hi = ci_d(diff, msk); r.update({f"d{c_}": diff[msk].mean(), f"lo{c_}": lo, f"hi{c_}": hi})
    r.update({cc: diff[cellv == cc].mean() for cc in ["K+C+", "K+C-", "K−C+", "K−C-"]})
    r.update(hit=d.hit_limit.mean(), rep3=d.rep3.mean())
    r["stable"] = r["hit"] <= .05 and r["rep3"] <= .05; r["eff"] = r["loC-"] > 0; r["safe"] = r["loC+"] > -0.02
    N = {"last": 1 + d.n_tok.values, "all": d.n_prompt.values + d.n_tok.values,
         "all_noBOS": d.n_prompt.values - 1 + d.n_tok.values, "decode": d.n_tok.values}[mode]
    sc = np.array([[rho * MED[k][L] for L in LAYERS] for k in d.fold.values])
    r.update(N_inj=(5 * N).mean(), D_sum=(N[:, None] * sc).sum(1).mean(), D_energy=np.sqrt((N[:, None] * sc**2).sum(1)).mean())
    rows.append(r)
T2 = pd.DataFrame(rows); T2.to_csv(OUT + "exp2_summary.csv", index=False)
pd.set_option("display.width", 250)
print(T2.round(4).to_string(index=False))

# ② nested 선택 (사전등록 규칙)
def nested(prefix):
    names = [n for n in E2 if n.startswith(prefix + "_r")]; pol, pick = a_em.astype(float).copy(), {}
    for k in range(5):
        oth, best = fo != k, (None, a_em[(fo != k) & isCm].mean())
        for n in names:
            d, e = E2[n], E2[n].em.values
            ok = d.hit_limit.values[oth].mean() <= .05 and d.rep3.values[oth].mean() <= .05 \
                 and (e - a_em)[oth & ~isCm].mean() >= -0.02
            if ok and e[oth & isCm].mean() > best[1]: best = (n, e[oth & isCm].mean())
        pick[k] = best[0]
        if best[0]: pol[fo == k] = E2[best[0]].em.values[fo == k]
    return pick, pol
NP = {}
for pre in ["P10", "P9", "CF"]:
    pick, pol = nested(pre); NP[pre] = pol; d = pol - a_em
    s = "  ".join(f"{c_} {d[msk].mean():+.4f} [{ci_d(d, msk)[0]:+.4f}, {ci_d(d, msk)[1]:+.4f}]" for c_, msk in [("C-", isCm), ("C+", ~isCm)])
    print(f"\n[nested {pre}] 선택 {pick}\n   {s}")
dd = NP["P9"] - NP["P10"]
print(f"\n[P9 − P10 nested, paired]  C− {dd[isCm].mean():+.4f} {np.round(ci_d(dd, isCm), 4)}  "
      f"C+ {dd[~isCm].mean():+.4f} {np.round(ci_d(dd, ~isCm), 4)}  전체 {dd.mean():+.4f} {np.round(ci_d(dd, np.ones(1224, bool)), 4)}")

# ③ C+ 손실 강도에서 게이트 재평가
G = pd.read_parquet(OUT + "exp1_gates.parquet")
print("\n[게이트 재평가]  cond / policy / open / C− / C+ / 전체 [CI] / vs always [CI]")
for n in ["P9_r0.09", "P9_r0.12", "P10_r0.15", "P10_r0.20"]:
    e = E2[n].em.values; alw = e - a_em
    for pn, op in [("always", np.ones(1224, bool)), ("gate_GF", G.gate_GF.values), ("gate_Kref", G.gate_Kref.values)]:
        d = np.where(op, e, a_em) - a_em; v = d - alw; all_ = np.ones(1224, bool)
        print(f"  {n:10s} {pn:9s} {op.mean():.2f}  {d[isCm].mean():+.4f}  {d[~isCm].mean():+.4f}  "
              f"{d.mean():+.4f} {np.round(ci_d(d, all_), 4)}  {v.mean():+.4f} {np.round(ci_d(v, all_), 4)}")
    print(f"  {n:10s} ceiling          {(np.maximum(e, a_em) - a_em).mean():+.4f}")

         cond  rho     dC-    loC-   hiC-     dC+    loC+    hiC+    K+C+    K+C-    K−C+    K−C-    hit   rep3  stable   eff  safe     N_inj     D_sum  D_energy
    P10_r0.03 0.03  0.0523  0.0359 0.0703  0.0016 -0.0082  0.0114  0.0029  0.0780  0.0000  0.0188 0.0000 0.0000    True  True  True   20.6127    6.1271    1.3428
     P9_r0.03 0.03  0.0833  0.0621 0.1062  0.0000 -0.0114  0.0114  0.0058  0.1214 -0.0075  0.0338 0.0000 0.0000    True  True  True 1223.4477  363.6675   10.5053
    P10_r0.06 0.06  0.0833  0.0621 0.1062 -0.0033 -0.0163  0.0098  0.0029  0.1243 -0.0113  0.0301 0.0000 0.0000    True  True  True   20.4085   12.1327    2.6725
     P9_r0.06 0.06  0.1356  0.1078 0.1634  0.0016 -0.0163  0.0180  0.0058  0.1994 -0.0038  0.0526 0.0000 0.0000    True  True  True 1223.4722  727.3496   21.0105
    P10_r0.09 0.09  0.1029  0.0801 0.1275  0.0033 -0.0115  0.0196  0.0116  0.1503 -0.0075  0.0414 0.0000 0.0000    True  True  True   20.5923   18.3631    4.0023
     P9_r0.09 0.09  0.1667  

In [ ]:
## Cell 15 — 최종 설정 동결 + Exp3 w 변형 (8조건 × 1224행: A100 약 8분 / L4 약 20분)
PREREG["frozen_config_after_exp2"] = {
 "date": dt.datetime.now(dt.timezone(dt.timedelta(hours=9))).isoformat(),
 "config": "P9 all non-pad prefill + decode, rho=0.06 x median_train||h_L||, L14-18, w_v3 procedure, no gate",
 "basis": "nested selection chose rho 0.06 in 5/5 folds; P10 nested failed held-out C+ safety; gates never beat always-on",
 "status": "exploratory on p8s (development set); external replication must use this config unchanged"}
json.dump(PREREG, open(OUT + "prereg_v3.json", "w"), ensure_ascii=False, indent=1)
print("동결 기록:", md5(OUT + "prereg_v3.json"))

scq = m.question.map(dict(zip(cb.question, cb.sc_v3))).values
ret, acc = (Lb.ret_x == 1).values, (Lb.acc_x == 1).values
def pbd_weighted(X, pos, neg, min_n=3):                         # prop별 차이를 조화평균 n으로 가중
    ds, ws = [], []
    for p_ in np.unique(PR[pos | neg]):
        i = PR == p_; a_, b_ = (pos & i).sum(), (neg & i).sum()
        if a_ >= min_n and b_ >= min_n: ds.append(X[pos & i].mean(0) - X[neg & i].mean(0)); ws.append(2 / (1 / a_ + 1 / b_))
    d = np.average(ds, axis=0, weights=ws); return d / np.linalg.norm(d)

VAR, VINFO = {}, []
for name in ["w_Kplus", "w_SC", "w_noprop", "w_nweight"]:
    VAR[name] = {}
    for k in range(5):
        tr = (Lb.fold != k).values & isC
        if name == "w_Kplus": tr &= (kv == "K+")                 # gold 사용 (명시)
        if name == "w_SC":    tr &= (scq >= 0.6)                 # gold 없음
        pos, neg = tr & ret, tr & acc
        if name == "w_noprop":
            f_ = lambda X: (lambda d: d / np.linalg.norm(d))(X[pos].mean(0) - X[neg].mean(0))
        elif name == "w_nweight":
            f_ = lambda X: pbd_weighted(X, pos, neg)
        else:
            f_ = lambda X: prop_bal_diff(X, pos, neg)[0]
        VAR[name][k] = {L: f_(HL[L]) for L in LAYERS}
        VINFO.append(dict(var=name, fold=k, n_ret=int(pos.sum()), n_acc=int(neg.sum()),
                          cos_w_v3_L16=round(float(VAR[name][k][16] @ W3[k][16]), 3)))
VI = pd.DataFrame(VINFO); print(VI.groupby("var")[["n_ret", "n_acc", "cos_w_v3_L16"]].mean().round(3))
np.savez(OUT + "w_variants.npz", **{f"{n}_f{k}_L{L}": VAR[n][k][L] for n in VAR for k in range(5) for L in LAYERS})

def run3(name, vec, mode, scale_fn):
    f = OUT + f"exp3_{name}.parquet"
    if os.path.exists(f): return pd.read_parquet(f)
    rows, t0 = [], time.time()
    for k in range(5):
        st2.set(vec[k], {L: scale_fn(k, L) for L in LAYERS}, mode); o = gen_fold2(k)
        assert all(st2.calls.get(L, 0) > 0 for L in LAYERS)
        rows += [dict(rid=r, raw=x, n_tok=n, n_prompt=npv, fold=k) for r, (x, n, npv) in o.items()]
    st2.set(None)
    d = score_v3(pd.DataFrame(rows).sort_values("rid").reset_index(drop=True)); d["cond"] = name
    d.to_parquet(f, index=False); print(f"{name:22s} {time.time()-t0:4.0f}s"); return d

CFG = {"P10_6sig": ("last", lambda k, L: 6.0 * SIG[k][L], V),              # 사전등록 주 설정 (w_v3 기준 = gen_v3_w+6)
       "P9_r0.06": ("all", lambda k, L: 0.06 * MED[k][L], E2["P9_r0.06"])}  # 동결 설정 (탐색적)
out = []
for cfg, (mode, sfn, ref) in CFG.items():
    for vn in VAR:
        d = run3(f"{cfg}_{vn}", VAR[vn], mode, sfn)
        for tag, dd in [(vn, d.em.values), ("w_v3(ref)", ref.em.values)]:
            diff = dd - a_em
            out.append(dict(cfg=cfg, var=tag, dC_m=diff[isCm].mean(), loC_m=ci_d(diff, isCm)[0],
                            dC_p=diff[~isCm].mean(), loC_p=ci_d(diff, ~isCm)[0],
                            vs_ref=(dd - ref.em.values)[isCm].mean(), vs_ref_lo=ci_d(dd - ref.em.values, isCm)[0],
                            vs_ref_hi=ci_d(dd - ref.em.values, isCm)[1]))
T3 = pd.DataFrame(out).drop_duplicates(["cfg", "var"]); T3.to_csv(OUT + "exp3_summary.csv", index=False)
print(T3.round(4).to_string(index=False))

동결 기록: 62f6a45328ab8fb6b24d4d066e33290c
           n_ret  n_acc  cos_w_v3_L16
var                                  
w_Kplus    112.0  106.4         0.960
w_SC       117.6  154.4         0.987
w_noprop   119.2  232.8         0.792
w_nweight  119.2  232.8         0.984
P10_6sig_w_Kplus         57s
P10_6sig_w_SC            61s
P10_6sig_w_noprop        57s
P10_6sig_w_nweight       70s
P9_r0.06_w_Kplus         53s
P9_r0.06_w_SC            58s
P9_r0.06_w_noprop        57s
P9_r0.06_w_nweight       53s
     cfg       var   dC_m  loC_m    dC_p   loC_p  vs_ref  vs_ref_lo  vs_ref_hi
P10_6sig   w_Kplus 0.1013 0.0768  0.0049 -0.0114  0.0016    -0.0065     0.0098
P10_6sig w_v3(ref) 0.0997 0.0768  0.0049 -0.0131  0.0000     0.0000     0.0000
P10_6sig      w_SC 0.0997 0.0768 -0.0033 -0.0212  0.0000    -0.0049     0.0049
P10_6sig  w_noprop 0.0964 0.0719  0.0049 -0.0114 -0.0033    -0.0147     0.0082
P10_6sig w_nweight 0.1046 0.0801  0.0033 -0.0131  0.0049     0.0000     0.0114
P9_r0.06   w_Kplus 0.1291 

In [ ]:
## Cell 16 — (a) w_allC: C 라벨 없는 w  (b) P9 ρ0.06에서 방향 특이성 (perm×10 + rand×3)
##          16조건 × 1224행: A100 약 15분 / L4 약 35분 (조건별 체크포인트)
W_ALLC, AI = {}, []
for k in range(5):
    tr = (Lb.fold != k).values                                    # C 구분 없이 전체 train 행
    pos, neg = tr & ret, tr & acc
    W_ALLC[k] = {L: prop_bal_diff(HL[L], pos, neg)[0] for L in LAYERS}
    AI.append(dict(fold=k, n_ret=int(pos.sum()), n_ret_Cp=int((pos & ~isC).sum()),
                   n_acc=int(neg.sum()), n_acc_Cp=int((neg & ~isC).sum()), cos_w_v3=round(float(W_ALLC[k][16] @ W3[k][16]), 3)))
print(pd.DataFrame(AI).to_string(index=False))
np.savez(OUT + "w_allC.npz", **{f"w_f{k}_L{L}": W_ALLC[k][L] for k in range(5) for L in LAYERS})

P9cfg = ("all", lambda k, L: 0.06 * MED[k][L])
R16 = {"P9_r0.06_w_allC": run3("P9_r0.06_w_allC", W_ALLC, *P9cfg),
       "P10_6sig_w_allC": run3("P10_6sig_w_allC", W_ALLC, "last", lambda k, L: 6.0 * SIG[k][L])}
for s in range(10): R16[f"P9_r0.06_perm{s}"] = run3(f"P9_r0.06_perm{s}", CONDS[f"perm{s}+6"], *P9cfg)
for s in range(3):  R16[f"P9_r0.06_rand{s}"] = run3(f"P9_r0.06_rand{s}", {k: RAND[s] for k in range(5)}, *P9cfg)

ref9 = E2["P9_r0.06"].em.values
rows = []
for n, d in {"P9_r0.06_w_v3(ref)": E2["P9_r0.06"], "P10_6sig_w_v3(ref)": V, **R16}.items():
    diff = d.em.values - a_em
    rows.append(dict(cond=n, dC_m=diff[isCm].mean(), loC_m=ci_d(diff, isCm)[0], dC_p=diff[~isCm].mean(),
                     loC_p=ci_d(diff, ~isCm)[0], hit=d.hit_limit.mean()))
T16 = pd.DataFrame(rows); T16.to_csv(OUT + "exp16_summary.csv", index=False)
print(T16.round(4).to_string(index=False))

dv = (ref9 - a_em)[isCm].mean()
null = np.array([(R16[n].em.values - a_em)[isCm].mean() for n in R16 if "perm" in n or "rand" in n])
permm = np.mean([(R16[f"P9_r0.06_perm{s}"].em.values - a_em)[isCm].mean() for s in range(10)])
print(f"\n[P9 ρ0.06 특이성] w_v3 {dv:+.4f} | null max {null.max():+.4f}  95th {np.percentile(null, 95):+.4f} | "
      f"w > null 전부 {(dv > null).all()} | perm 평균 {permm:+.4f} → 특이적 효과 ≈ {dv - permm:+.4f}")
for n in ["P9_r0.06_w_allC", "P10_6sig_w_allC"]:
    ref = ref9 if n.startswith("P9") else V.em.values; dd = R16[n].em.values - ref
    print(f"[{n}] vs w_v3  C− {dd[isCm].mean():+.4f} {np.round(ci_d(dd, isCm), 4)}  C+ {dd[~isCm].mean():+.4f} {np.round(ci_d(dd, ~isCm), 4)}")

 fold  n_ret  n_ret_Cp  n_acc  n_acc_Cp  cos_w_v3
    0    163        45    559       325     0.960
    1    159        39    572       331     0.957
    2    160        37    554       331     0.970
    3    169        48    547       319     0.956
    4    153        39    568       330     0.927
P9_r0.06_w_allC          55s
P10_6sig_w_allC          61s
P9_r0.06_perm0           57s
P9_r0.06_perm1           54s
P9_r0.06_perm2           55s
P9_r0.06_perm3           55s
P9_r0.06_perm4           55s
P9_r0.06_perm5           56s
P9_r0.06_perm6           58s
P9_r0.06_perm7           54s
P9_r0.06_perm8           56s
P9_r0.06_perm9           56s
P9_r0.06_rand0           57s
P9_r0.06_rand1           56s
P9_r0.06_rand2           55s
              cond    dC_m   loC_m    dC_p   loC_p  hit
P9_r0.06_w_v3(ref)  0.1356  0.1078  0.0016 -0.0163  0.0
P10_6sig_w_v3(ref)  0.0997  0.0768  0.0049 -0.0131  0.0
   P9_r0.06_w_allC  0.1209  0.0948 -0.0016 -0.0212  0.0
   P10_6sig_w_allC  0.0899  0.0670 -0.001

In [ ]:
## Cell 17 — (a) w_allC_dis: 충돌 행만(CB≠컨텍스트, gold·C 라벨 없음) [탐색적]
##           (b) gate_EM nested (사전등록 완결, P10 +6)
##           (a) 2조건 + (b) inner 생성 20회(≈4×1224행): A100 약 7분 / L4 약 17분
s_n = m.answer_surface.map(PP.norm).values
c_n = m.question.map(cbq).map(PP.norm).values
dis = (c_n != "") & (c_n != s_n)
W_DIS = {}
for k in range(5):
    tr = (fo != k) & dis
    W_DIS[k] = {L: prop_bal_diff(HL[L], tr & ret, tr & acc)[0] for L in LAYERS}
print("충돌 행 수 C−/C+:", int((dis & isC).sum()), int((dis & ~isC).sum()),
      "| cos(w_dis, w_v3) L16:", [round(float(W_DIS[k][16] @ W3[k][16]), 3) for k in range(5)])
R17 = {"P9_r0.06_w_allCdis": run3("P9_r0.06_w_allCdis", W_DIS, "all", lambda k, L: 0.06 * MED[k][L]),
       "P10_6sig_w_allCdis": run3("P10_6sig_w_allCdis", W_DIS, "last", lambda k, L: 6.0 * SIG[k][L])}
for n, d in R17.items():
    ref = ref9 if n.startswith("P9") else V.em.values
    dd, da = d.em.values - ref, d.em.values - a_em
    print(f"[{n}] ΔEM C− {da[isCm].mean():+.4f}  C+ {da[~isCm].mean():+.4f} (하한 {ci_d(da, ~isCm)[0]:+.4f}) | "
          f"vs w_v3 C− {dd[isCm].mean():+.4f} {np.round(ci_d(dd, isCm), 4)}")

# (b) gate_EM nested: τ는 outer-train 안의 inner cross-fitting 출력으로만 선택
def fit_gate(trm):
    mu, sd = H13[trm].mean(0), H13[trm].std(0) + 1e-6; Z = (H13 - mu) / sd
    cd = Z[trm & (ysc == 1)].mean(0) - Z[trm & (ysc == 0)].mean(0); return Z @ (cd / np.linalg.norm(cd))
fI = OUT + "exp1_gateEM_inner.parquet"
if os.path.exists(fI): I = pd.read_parquet(fI)
else:
    rec, t0 = [], time.time()
    for k in range(5):
        for j in [x for x in range(5) if x != k]:
            itr = (fo != k) & (fo != j)
            wv = {L: prop_bal_diff(HL[L], itr & isC & ret, itr & isC & acc)[0] for L in LAYERS}
            sg = {L: float(HL[L][itr].std()) for L in LAYERS}
            st2.set(wv, {L: 6.0 * sg[L] for L in LAYERS}, "last"); o = gen_fold2(j); st2.set(None)
            gs = fit_gate(itr)
            rec += [dict(outer=k, inner=j, rid=r, raw=x, gs=float(gs[r])) for r, (x, n, npv) in o.items()]
    I = pd.DataFrame(rec)
    I["em"] = [int(any(PP.norm(trunc_v3(x)) == PP.norm(g) for g in gl_row[r])) for r, x in zip(I.rid, I.raw)]
    I.to_parquet(fI, index=False); print(f"inner 생성 {time.time()-t0:.0f}s")

open_em = np.zeros(1224, bool); TAU = {}
for k in range(5):
    Ik = I[I.outer == k]; ae = a_em[Ik.rid.values]
    cand = np.quantile(Ik.gs, np.linspace(0, 1, 101))
    sc_ = [np.where(Ik.gs.values > t, Ik.em.values, ae).mean() for t in cand]
    TAU[k] = float(cand[int(np.argmax(sc_))]); te = fo == k
    open_em[te] = gscore[te] > TAU[k]
pe = np.where(open_em, V.em.values, a_em); d, v = pe - a_em, pe - V.em.values
print(f"\n[gate_EM nested] τ {np.round(list(TAU.values()), 3)}  개방률 {open_em.mean():.2f}")
print(f"  C− {d[isCm].mean():+.4f}  C+ {d[~isCm].mean():+.4f}  전체 {d.mean():+.4f} {np.round(ci_d(d, np.ones(1224, bool)), 4)}"
      f"  | vs always {v.mean():+.4f} {np.round(ci_d(v, np.ones(1224, bool)), 4)}")

충돌 행 수 C−/C+: 612 342 | cos(w_dis, w_v3) L16: [0.976, 0.976, 0.984, 0.987, 0.955]
P9_r0.06_w_allCdis       60s
P10_6sig_w_allCdis       58s
[P9_r0.06_w_allCdis] ΔEM C− +0.1307  C+ -0.0016 (하한 -0.0196) | vs w_v3 C− -0.0049 [-0.0114  0.    ]
[P10_6sig_w_allCdis] ΔEM C− +0.0948  C+ +0.0000 (하한 -0.0163) | vs w_v3 C− -0.0049 [-0.0131  0.0016]
inner 생성 252s

[gate_EM nested] τ [-28.712 -35.99  -34.026 -28.708 -24.356]  개방률 0.98
  C− +0.0980  C+ +0.0049  전체 +0.0515 [0.0367 0.0662]  | vs always -0.0008 [-0.0025  0.    ]


In [ ]:
## Cell 18 — 주입량 눈금 병기표: σ_grand · σ_proj · ρ · RMS · 표준 d′ (CPU, 즉시)
rows = []
for k in range(5):
    tr, trC = fo != k, (fo != k) & isC
    for L in LAYERS:
        w_ = W3[k][L]; proj = HL[L] @ w_
        s_g, s_p, med = SIG[k][L], float(proj[tr].std()), MED[k][L]
        pr_, pa_ = proj[trC & ret], proj[trC & acc]
        gap = pr_.mean() - pa_.mean(); d_std = gap / np.sqrt((pr_.var() + pa_.var()) / 2)
        pdm = proj.copy()                                           # prop 평균 제거 버전
        for p_ in np.unique(PR[trC]): i = trC & (PR == p_); pdm[i] -= proj[i].mean()
        g2 = pdm[trC & ret].mean() - pdm[trC & acc].mean()
        d_pdm = g2 / np.sqrt((pdm[trC & ret].var() + pdm[trC & acc].var()) / 2)
        for cfg, dh in [("P10_legacy+6σ", 6.0 * s_g), ("P9_frozen_ρ0.06", 0.06 * med)]:
            rows.append(dict(cfg=cfg, fold=k, L=L, dh=dh, coef_grand=dh / s_g, coef_proj=dh / s_p, rho=dh / med,
                             dh_over_RMScoord=dh / (med / 64), sig_proj_over_grand=s_p / s_g,
                             gap=gap, d_prime=d_std, gap_z=gap / s_p, d_prime_propdemeaned=d_pdm, shift_over_gap=dh / gap))
S18 = pd.DataFrame(rows); S18.to_csv(OUT + "dose_scale_table.csv", index=False)
print(S18.groupby(["cfg", "L"]).mean(numeric_only=True).drop(columns="fold").round(3).to_string())

                       dh  coef_grand  coef_proj    rho  dh_over_RMScoord  sig_proj_over_grand    gap  d_prime  gap_z  d_prime_propdemeaned  shift_over_gap
cfg             L                                                                                                                                          
P10_legacy+6σ   14  0.764       6.000      1.163  0.094             5.998                5.169  0.901    1.695  1.370                 1.317           0.848
                15  0.816       6.000      0.695  0.094             5.996                8.635  1.682    1.993  1.433                 1.485           0.485
                16  0.898       6.000      0.667  0.093             5.972                8.998  1.969    2.000  1.462                 1.561           0.457
                17  1.014       6.000      0.665  0.094             5.994                9.037  2.332    2.218  1.527                 1.661           0.435
                18  1.144       6.000      0.673  0.094         

### 실험 피드백 후 추가 실험

In [ ]:
## Cell 19 — 사전등록 파일 분리 + 원본 해시 복원 검증 (CPU)
now = lambda: dt.datetime.now(dt.timezone(dt.timedelta(hours=9))).isoformat()
orig = {k: v for k, v in PREREG.items() if k not in ("gate_training_rows", "exp2", "frozen_config_after_exp2")}
p0 = OUT + "prereg_v3_original.json"
json.dump(orig, open(p0, "w"), ensure_ascii=False, indent=1)
h0 = md5(p0); print("원본 복원", h0, "| Cell 10 기록과 일치:", h0 == "06a8fbcb1eb887c77cbab66648f4860b")

AMD = {
 "amendment_gate_rows_preoutcome.json": {
    "type": "pre-outcome protocol amendment (사전등록 누락 보완)", "base_prereg_md5": h0,
    "content": PREREG["gate_training_rows"],
    "when": "Cell 11 첫 줄, 게이트 결과 계산 전 기록 (~2026-09-22 00:05 KST). 별도 타임스탬프는 남기지 않음",
    "results_seen_before": "v3 anchor(Cell 10)만. 게이트 결과 없음"},
 "prereg_exp2.json": {
    "type": "Exp2 실행 전 프로토콜", "base_prereg_md5": h0, "content": PREREG["exp2"],
    "combined_file_md5_at_save": "1c984cecb93d5f6fedd2b576ed5c541d",
    "results_seen_before": "Exp1, P10+6 방향 특이성",
    "correction": f"{now()} — 선택은 held-out fold가 다른 fold의 w 학습에 쓰인 OOF 출력으로 수행됨. 엄밀한 nested가 아니라 cross-fitted selection으로 명칭 정정"},
 "exploratory_w_allCdis.json": {
    "type": "post-hoc exploratory", "defined_after": "w_allC 결과(Cell 16) 확인 후",
    "definition": "전체 train 행 중 CB답 != 컨텍스트답, retain=ret_x / accept=acc_x, prop 균형",
    "status": "p8s에서 확인적 결과 아님. 외부 데이터에서 사전등록 조건으로 검증"},
 "external_validation_protocol_v1.json": {
    "type": "외부검증 프로토콜 (p8s 탐색 결과로 동결한 설정)", "created": now(),
    "frozen_config": PREREG["frozen_config_after_exp2"],
    "dataset": "미정 — 어떤 외부 데이터 생성보다 먼저 확정해 이 파일을 갱신(v2)", "status": "draft"}}
for f, v in AMD.items():
    json.dump(v, open(OUT + f, "w"), ensure_ascii=False, indent=1); print(f"{f:42s} {md5(OUT + f)}")
os.rename(OUT + "prereg_v3.json", OUT + "prereg_v3_combined_overwritten.json")
print("prereg_v3.json → prereg_v3_combined_overwritten.json (보존, 인용 금지)")

원본 복원 06a8fbcb1eb887c77cbab66648f4860b | Cell 10 기록과 일치: True
amendment_gate_rows_preoutcome.json        3077570c32e40e001ab71a4ce7fa0161
prereg_exp2.json                           4b611c44bef1d2fb7c80ae8310fc4bba
exploratory_w_allCdis.json                 560547e0d89eb3940497f6df2db1408a
external_validation_protocol_v1.json       0a22ddafb7aa8ebba579e0aa2b2e5ffe
prereg_v3.json → prereg_v3_combined_overwritten.json (보존, 인용 금지)


In [ ]:
## Cell 20 — 배치 민감도: 동결 설정(P9 ρ0.06)을 다른 배치로 (4조건: A100 약 4분 / L4 약 10분)
ALT = {"bs32_len": (32, lambda k: FOLD_ROWS[k]),
       "bs8_rid":  (8,  lambda k: m[m.fold == k].rid.values)}          # 길이 정렬 없이 rid 순서
@torch.no_grad()
def gen_alt(rids, bs):
    out = {}
    for s in range(0, len(rids), bs):
        b = rids[s:s+bs]
        enc = tok(m.prompt.values[b].tolist(), return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        st2.mask = enc.attention_mask.clone()
        g = model.generate(**enc, max_new_tokens=PP.MAX_NEW_TOKENS, do_sample=False,
                           pad_token_id=tok.pad_token_id, eos_token_id=EOS_IDS)[:, enc.input_ids.shape[1]:]
        for r, row in zip(b, g.tolist()):
            stop = next((i for i, t in enumerate(row) if t in EOS_IDS), None)
            n = stop if stop is not None else len(row); out[int(r)] = (tok.decode(row[:n], **DECODE_KW), n)
    return out

for tag, (bs, rows_fn) in ALT.items():
    res = {}
    for cond, vec in [("base", None), ("P9_r0.06", W3)]:
        f = OUT + f"batch_{tag}_{cond}.parquet"
        if os.path.exists(f): res[cond] = pd.read_parquet(f); continue
        rr_ = []
        for k in range(5):
            st2.set(None if vec is None else vec[k], None if vec is None else {L: 0.06 * MED[k][L] for L in LAYERS}, "all")
            rr_ += [dict(rid=r, raw=x, n_tok=n) for r, (x, n) in gen_alt(rows_fn(k), bs).items()]
        st2.set(None)
        res[cond] = score_v3(pd.DataFrame(rr_).sort_values("rid").reset_index(drop=True)); res[cond].to_parquet(f, index=False)
    mb = (res["base"].raw.values != B.raw.values).sum(); ms = (res["P9_r0.06"].raw.values != E2["P9_r0.06"].raw.values).sum()
    d = res["P9_r0.06"].em.values - res["base"].em.values
    print(f"[{tag}] 주 배치 대비 문자열 불일치 base {mb} / steer {ms}  |  "
          f"C− {d[isCm].mean():+.4f} {np.round(ci_d(d, isCm), 4)}  C+ {d[~isCm].mean():+.4f} {np.round(ci_d(d, ~isCm), 4)}")
print("[주 배치 bs16_len] C− +0.1356 [0.1078 0.1634]  C+ +0.0016 [-0.0163 0.018]")

[bs32_len] 주 배치 대비 문자열 불일치 base 10 / steer 12  |  C− +0.1340 [0.1062 0.1618]  C+ +0.0016 [-0.0163  0.0196]
[bs8_rid] 주 배치 대비 문자열 불일치 base 11 / steer 16  |  C− +0.1340 [0.1062 0.1618]  C+ -0.0016 [-0.0196  0.0147]
[주 배치 bs16_len] C− +0.1356 [0.1078 0.1634]  C+ +0.0016 [-0.0163 0.018]


In [ ]:
## Cell 21 — (a) held-out d′  (b) 실제 이동량 측정: steering 전후 L13–20 마지막 토큰의 w축 이동
##          prefill만 612행 × 3회 (A100 약 1분 / L4 약 3분)
# (a) held-out: fold k의 test C− 행을 W3[k]에 투영
R21 = []
for L in LAYERS:
    pr_, pa_ = [], []
    for k in range(5):
        te, p_ = (fo == k) & isC, HL[L] @ W3[k][L]
        pr_.append(p_[te & ret]); pa_.append(p_[te & acc])
    pr_, pa_ = np.concatenate(pr_), np.concatenate(pa_); gap = pr_.mean() - pa_.mean()
    R21.append(dict(L=L, gap_heldout=gap, d_prime_heldout=gap / np.sqrt((pr_.var() + pa_.var()) / 2),
                    gap_insample=S18[(S18.cfg == "P9_frozen_ρ0.06") & (S18.L == L)].gap.mean(),
                    d_prime_insample=S18[(S18.cfg == "P9_frozen_ρ0.06") & (S18.L == L)].d_prime.mean()))
R21 = pd.DataFrame(R21).set_index("L"); print(R21.round(3).to_string())

# (b) 실제 이동량
LCAP, CAP = list(range(13, 21)), {}
def mk_cap(L):
    def hook(mod, args, out):
        h = out[0] if isinstance(out, (tuple, list)) else out; CAP[L] = h[:, -1, :].float().cpu().numpy()
    return hook
caps = [model.model.layers[L].register_forward_hook(mk_cap(L)) for L in LCAP]    # st2 훅 뒤에 등록 → 주입 후 값을 봄

@torch.no_grad()
def prefill_caps(rids):
    out = {L: [] for L in LCAP}
    for s in range(0, len(rids), BS):
        b = rids[s:s+BS]
        enc = tok(m.prompt.values[b].tolist(), return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        st2.mask = enc.attention_mask.clone()
        model.model(input_ids=enc.input_ids, attention_mask=enc.attention_mask, use_cache=False)
        for L in LCAP: out[L].append(CAP[L])
    return {L: np.concatenate(v) for L, v in out.items()}

acc_rows = []
for k in range(5):
    rids = np.array([r for r in FOLD_ROWS[k] if isCm[r]])
    st2.set(None); hb = prefill_caps(rids)
    for cfg, mode, sfn in [("P9_ρ0.06", "all", lambda L: 0.06 * MED[k][L]), ("P10_+6σ", "last", lambda L: 6.0 * SIG[k][L])]:
        st2.set(W3[k], {L: sfn(L) for L in LAYERS}, mode); hs = prefill_caps(rids); st2.set(None)
        cum = 0.0
        for L in LCAP:
            wL = W3[k][min(max(L, 14), 18)]
            if 14 <= L <= 18: cum += sfn(L)
            dH = hs[L] - hb[L]
            acc_rows.append(dict(cfg=cfg, fold=k, L=L, dproj=(dH @ wL).mean(), dnorm=np.linalg.norm(dH, axis=1).mean(),
                                 linear_pred=cum))
for c_ in caps: c_.remove()

A21 = pd.DataFrame(acc_rows).groupby(["cfg", "L"]).mean(numeric_only=True).drop(columns="fold")
A21["actual/linear"] = A21.dproj / A21.linear_pred.replace(0, np.nan)
A21["dproj/gap_heldout"] = [r.dproj / R21.gap_heldout.get(min(max(L, 14), 18)) for (c, L), r in A21.iterrows()]
A21["along_w_share"] = A21.dproj / A21.dnorm
print(A21.round(3).to_string()); A21.to_csv(OUT + "displacement_measured.csv"); R21.to_csv(OUT + "dprime_heldout.csv")

    gap_heldout  d_prime_heldout  gap_insample  d_prime_insample
L                                                               
14        0.854            1.616         0.901             1.695
15        1.599            1.885         1.682             1.993
16        1.889            1.927         1.969             2.000
17        2.228            2.136         2.332             2.218
18        2.481            2.100         2.623             2.181
             dproj  dnorm  linear_pred  actual/linear  dproj/gap_heldout  along_w_share
cfg      L                                                                             
P10_+6σ  13  0.000  0.000        0.000            NaN              0.000            NaN
         14  0.764  0.764        0.764          1.000              0.895          1.000
         15  1.272  1.398        1.580          0.805              0.795          0.910
         16  2.068  2.330        2.478          0.835              1.095          0.888
         17  2.94